In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import os
import random
import csv
import requests
from collections import Counter

# Config & Reproducibility

SEED = 42
MODEL_NAME = 'GroNLP/hateBERT'
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
LAMBDA_RAT = 1.0
SAVE_DIR = "/content/gdrive/MyDrive/models/CAP_model"
LOG_PATH = os.path.join(SAVE_DIR, "training_log.csv")

LABEL2ID = {"normal": 0, "offensive": 1, "hatespeech": 2}
NUM_LABELS = len(LABEL2ID)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
# Download & Split HateXplain Dataset

BASE_URL = "https://raw.githubusercontent.com/punyajoy/HateXplain/master/Data/"

print("Downloading HateXplain dataset...")
dataset = requests.get(BASE_URL + "dataset.json").json()

TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
MIN_ANNOTATOR_AGREEMENT = 2

all_post_ids = list(dataset.keys())
random.seed(SEED)
random.shuffle(all_post_ids)

total = len(all_post_ids)
train_end = int(total * TRAIN_RATIO)
val_end   = train_end + int(total * VAL_RATIO)

train_ids = all_post_ids[:train_end]
val_ids   = all_post_ids[train_end:val_end]
test_ids  = all_post_ids[val_end:]

def load_ids_to_df(id_list):
    rows = []
    for tweet_id in id_list:
        info = dataset[tweet_id]
        post_tokens = info["post_tokens"]

        label_counts = Counter(a["label"] for a in info["annotators"])
        if not label_counts:
            continue

        final_label = label_counts.most_common(1)[0][0]
        if final_label not in LABEL2ID:
            continue

        if MIN_ANNOTATOR_AGREEMENT > 1:
            majority_count = sum(1 for a in info["annotators"] if a["label"] == final_label)
            if majority_count < MIN_ANNOTATOR_AGREEMENT:
                continue

        consensus_rationale = [0] * len(post_tokens)

        if "rationales" in info and info["rationales"]:
            try:
                rationale_matrix = np.array(info["rationales"])
                n_annotators_with_rationales = np.sum(np.any(rationale_matrix, axis=1))
                if n_annotators_with_rationales > 0:
                    token_sums = np.sum(rationale_matrix, axis=0)
                    threshold = 0.5 * n_annotators_with_rationales
                    consensus_rationale = [1 if s >= threshold else 0 for s in token_sums]
            except ValueError:
                pass

        rows.append({
            "post_id": tweet_id,
            "majority_label": final_label,
            "post_tokens": post_tokens,
            "consensus_rationale": consensus_rationale
        })

    return pd.DataFrame(rows)

print("Processing splits into DataFrames...")
df_train = load_ids_to_df(train_ids)
df_val   = load_ids_to_df(val_ids)
df_test  = load_ids_to_df(test_ids)

def verify_alignment(df):
    bad_idx = []
    for i, row in df.iterrows():
        if len(row['post_tokens']) != len(row['consensus_rationale']):
            bad_idx.append(i)
    if bad_idx:
        df = df.drop(index=bad_idx).reset_index(drop=True)
    return df

def drop_empty_rows(df):
    df = df[df['post_tokens'].apply(len) > 0].reset_index(drop=True)
    return df

df_train = drop_empty_rows(verify_alignment(df_train))
df_val   = drop_empty_rows(verify_alignment(df_val))
df_test  = drop_empty_rows(verify_alignment(df_test))

train_words   = df_train['post_tokens'].tolist()
train_rats    = df_train['consensus_rationale'].tolist()
train_labels  = [LABEL2ID[lbl] for lbl in df_train['majority_label'].tolist()]

val_words     = df_val['post_tokens'].tolist()
val_rats      = df_val['consensus_rationale'].tolist()
val_labels    = [LABEL2ID[lbl] for lbl in df_val['majority_label'].tolist()]

test_words    = df_test['post_tokens'].tolist()
test_rats     = df_test['consensus_rationale'].tolist()
test_labels   = [LABEL2ID[lbl] for lbl in df_test['majority_label'].tolist()]

print(f"Final Count -> Train: {len(train_labels)} | Val: {len(val_labels)} | Test: {len(test_labels)}")

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)



Processing splits into DataFrames...
Final Count -> Train: 15377 | Val: 1929 | Test: 1923


In [ ]:
# Token-Aligned Dataset

class TokenAlignedDataset(Dataset):
    def __init__(self, words_list, rationales_list, labels, tokenizer, max_length=MAX_LENGTH):
        self.words_list = words_list
        self.rationales_list = rationales_list
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        words = self.words_list[idx]
        word_rationales = self.rationales_list[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        word_ids = encoding.word_ids(batch_index=0)

        token_valid_mask = torch.zeros(self.max_length, dtype=torch.float)
        token_rationale_mask = torch.zeros(self.max_length, dtype=torch.float)

        for seq_idx, w_id in enumerate(word_ids):
            # Ignore [CLS], [SEP], and [PAD] (where w_id is None)
            if w_id is not None:
                token_valid_mask[seq_idx] = 1.0
                if w_id < len(word_rationales):
                    token_rationale_mask[seq_idx] = float(word_rationales[w_id])

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long),
            'token_valid_mask': token_valid_mask,
            'token_rationale_mask': token_rationale_mask
        }



In [ ]:
'''
---------------------------------------------------------
CAP Architecture: Single Shared Linear Head
---------------------------------------------------------
One linear layer maps each token's hidden state to per-token,
per-class logits. Classification and rationale supervision are
two READOUTS OF THE SAME LOGITS, not two separately-parameterized
heads:

  token_logits[i, c]  = head(h_i)[c]                    (shared)
  sequence_logits[c]  = mean_i( token_logits[i, c] )     (softmax -> classification)
  rationale_logit[i]  = token_logits[i, y]               (sigmoid -> token rationale, class y)

Because sequence_logits is a plain mean of token_logits, each
token's rationale score (sigmoid of its own logit at the target
class) is a monotonic transform of that token's exact additive
contribution to the classification decision -- faithfulness by
construction, not by auxiliary loss alignment.
---------------------------------------------------------
'''

class CAPModel(nn.Module):
    def __init__(self, model_name, num_labels=3, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.num_labels = num_labels

        self.dropout = nn.Dropout(dropout)
        # SINGLE SHARED HEAD -- the only linear layer producing
        # class-relevant logits anywhere in the model.
        self.head = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_valid_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        subword_states = self.dropout(outputs.last_hidden_state)  # [B, L, H]

        # Shared head applied per-token -> per-token, per-class logits
        token_logits = self.head(subword_states)  # [B, L, C]

        # Zero out special/padding token logits before pooling
        valid_mask = token_valid_mask.unsqueeze(-1)  # [B, L, 1]
        masked_token_logits = token_logits * valid_mask

        # --- Classification readout: mean-pool over valid tokens ---
        valid_counts = token_valid_mask.sum(dim=1, keepdim=True).clamp(min=1e-9)  # [B, 1]
        sequence_logits = masked_token_logits.sum(dim=1) / valid_counts  # [B, C]

        # Return raw per-token, per-class logits; the rationale
        # readout (sigmoid at a specific class index) is computed
        # by the caller, since which class to index depends on
        # gold label (training) vs. predicted label (inference).
        return sequence_logits, token_logits




In [ ]:
'''
---------------------------------------------------------
Rationale Loss (reads out the SAME logits used for
   classification, indexed at the gold class)
---------------------------------------------------------
'''

def token_rationale_loss(token_logits, labels, token_rationale_mask, token_valid_mask):
    """
    token_logits: [B, L, C] -- shared-head logits (same tensor used
                  to compute sequence_logits via mean-pooling)
    labels:       [B]       -- gold class index per example
    """
    B, L, C = token_logits.shape

    rationale_sum = token_rationale_mask.sum(dim=1)
    has_rationale = (rationale_sum > 0)

    if has_rationale.sum() == 0:
        return torch.tensor(0.0, device=token_logits.device)

    # Gather each token's logit at the GOLD class -> [B, L]
    gold_idx = labels.view(B, 1, 1).expand(B, L, 1)
    gold_token_logits = token_logits.gather(dim=2, index=gold_idx).squeeze(-1)  # [B, L]

    pred_logits = gold_token_logits[has_rationale]
    targets = token_rationale_mask[has_rationale]
    valid_tokens = token_valid_mask[has_rationale]

    bce_loss_fn = nn.BCEWithLogitsLoss(reduction='none')
    loss = bce_loss_fn(pred_logits, targets)
    masked_loss = loss * valid_tokens

    denom = valid_tokens.sum()
    if denom == 0:
        return torch.tensor(0.0, device=token_logits.device)

    return masked_loss.sum() / denom




In [ ]:
#Evaluate taring just to pick the best model

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    token_f1_scores, iou_scores = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            token_valid_mask = batch['token_valid_mask'].to(device)
            gold_rationale = batch['token_rationale_mask'].to(device)

            sequence_logits, token_logits = model(input_ids, attention_mask, token_valid_mask)

            preds = torch.argmax(sequence_logits, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

            # Read out rationale at the PREDICTED class (no oracle leakage)
            B, L, C = token_logits.shape
            pred_idx = preds.view(B, 1, 1).expand(B, L, 1)
            pred_token_logits = token_logits.gather(dim=2, index=pred_idx).squeeze(-1)  # [B, L]
            token_probs = torch.sigmoid(pred_token_logits)
            pred_rationale_mask = (token_probs > 0.5).float()

            for i in range(input_ids.size(0)):
                gold = gold_rationale[i]
                tm = token_valid_mask[i]

                if gold.sum().item() == 0:
                    continue

                valid = tm.bool()
                p_mask = pred_rationale_mask[i][valid].bool()
                g_mask = gold[valid].bool()

                tp = (p_mask & g_mask).sum().item()
                fp = (p_mask & ~g_mask).sum().item()
                fn = (~p_mask & g_mask).sum().item()

                precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
                token_f1_scores.append(f1)

                intersection = (p_mask & g_mask).sum().item()
                union = (p_mask | g_mask).sum().item()
                iou = (intersection / union) if union > 0 else (1.0 if g_mask.sum().item() == 0 else 0.0)
                iou_scores.append(iou)

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    mean_token_f1 = float(np.mean(token_f1_scores)) if token_f1_scores else 0.0
    mean_iou = float(np.mean(iou_scores)) if iou_scores else 0.0

    return acc, macro_f1, mean_token_f1, mean_iou, all_preds, all_labels




In [ ]:
# Setup & Training Loop

print(f"Loading {MODEL_NAME} tokenizer and initializing CAP model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = CAPModel(MODEL_NAME, num_labels=NUM_LABELS).to(device)

train_dataset = TokenAlignedDataset(train_words, train_rats, train_labels, tokenizer)
val_dataset   = TokenAlignedDataset(val_words, val_rats, val_labels, tokenizer)
test_dataset  = TokenAlignedDataset(test_words, test_rats, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

cls_loss_fn = nn.CrossEntropyLoss(weight=class_weights)

with open(LOG_PATH, 'w', newline='') as f:
    csv.writer(f).writerow(['epoch', 'train_loss', 'cls_loss', 'rat_loss', 'val_acc', 'val_macro_f1', 'val_token_f1', 'val_iou'])

best_val_f1 = -1.0
best_model_path = os.path.join(SAVE_DIR, "best_cap_model.pt")

print(f"Starting CAP training on {device}...")

for epoch in range(EPOCHS):
    model.train()
    total_loss, total_cls_loss, total_rat_loss = 0.0, 0.0, 0.0
    n_valid_batches = 0

    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        token_valid_mask = batch['token_valid_mask'].to(device)
        token_rationale_mask = batch['token_rationale_mask'].to(device)

        optimizer.zero_grad()

        sequence_logits, token_logits = model(input_ids, attention_mask, token_valid_mask)

        cls_loss = cls_loss_fn(sequence_logits, labels)
        rat_loss = token_rationale_loss(token_logits, labels, token_rationale_mask, token_valid_mask)
        loss = cls_loss + (LAMBDA_RAT * rat_loss)

        if not torch.isfinite(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        total_cls_loss += cls_loss.item()
        total_rat_loss += rat_loss.item()
        n_valid_batches += 1

        loop.set_description(f'Epoch {epoch+1}')
        loop.set_postfix(Loss=loss.item(), Cls=cls_loss.item(), RatBCE=rat_loss.item())

    denom = max(n_valid_batches, 1)
    avg_loss = total_loss / denom
    avg_cls = total_cls_loss / denom
    avg_rat = total_rat_loss / denom

    val_acc, val_macro_f1, val_token_f1, val_iou, _, _ = evaluate(model, val_loader)

    print(f"\nEpoch {epoch+1} | Loss: {avg_loss:.4f} | Cls: {avg_cls:.4f} | RatBCE: {avg_rat:.4f}")
    print(f"Validation -> Acc: {val_acc:.4f} | Macro-F1: {val_macro_f1:.4f} | Token-F1: {val_token_f1:.4f} | IOU: {val_iou:.4f}\n")

    with open(LOG_PATH, 'a', newline='') as f:
        csv.writer(f).writerow([epoch+1, avg_loss, avg_cls, avg_rat, val_acc, val_macro_f1, val_token_f1, val_iou])

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"New best model saved based on Macro-F1: {val_macro_f1:.4f}")

print("Training complete!")

# Evaluation

model.load_state_dict(torch.load(best_model_path))
test_acc, test_macro_f1, test_token_f1, test_iou, test_preds, test_labels_out = evaluate(model, test_loader)

print(f"\n" + "="*60)
print(f"FINAL TEST SET RESULTS (CAP: Single Shared Linear Head)")
print(f"="*60)
print(f"Sentence Accuracy      : {test_acc:.4f}")
print(f"Sentence Macro-F1      : {test_macro_f1:.4f}")
print(f"Subword-Token F1       : {test_token_f1:.4f}")
print(f"Subword-Token IOU      : {test_iou:.4f}")
print(f"="*60)
print(classification_report(test_labels_out, test_preds, target_names=list(LABEL2ID.keys()), digits=4))


Loading GroNLP/hateBERT tokenizer and initializing CAP model...


config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...


Epoch 1 | Loss: 1.3774 | Cls: 0.8308 | RatBCE: 0.5466
Validation -> Acc: 0.6947 | Macro-F1: 0.6854 | Token-F1: 0.6894 | IOU: 0.5934

New best model saved based on Macro-F1: 0.6854


Epoch 2: 100%|██████████| 962/962 [06:03<00:00,  2.64it/s, Cls=0.552, Loss=0.552, RatBCE=0]



Epoch 2 | Loss: 1.0657 | Cls: 0.6222 | RatBCE: 0.4435
Validation -> Acc: 0.7009 | Macro-F1: 0.6963 | Token-F1: 0.7247 | IOU: 0.6264

New best model saved based on Macro-F1: 0.6963


Epoch 3: 100%|██████████| 962/962 [06:05<00:00,  2.63it/s, Cls=0.572, Loss=1.45, RatBCE=0.875]



Epoch 3 | Loss: 0.9052 | Cls: 0.5030 | RatBCE: 0.4022
Validation -> Acc: 0.7014 | Macro-F1: 0.6970 | Token-F1: 0.7241 | IOU: 0.6272

New best model saved based on Macro-F1: 0.6970
Training complete!

FINAL TEST SET RESULTS (CAP: Single Shared Linear Head)
Sentence Accuracy      : 0.6937
Sentence Macro-F1      : 0.6842
Subword-Token F1       : 0.6953
Subword-Token IOU      : 0.5992
              precision    recall  f1-score   support

      normal     0.7782    0.7238    0.7500       800
   offensive     0.5304    0.5460    0.5380       544
  hatespeech     0.7399    0.7910    0.7646       579

    accuracy                         0.6937      1923
   macro avg     0.6828    0.6869    0.6842      1923
weighted avg     0.6966    0.6937    0.6944      1923



# Evaluation

In [ ]:
#!/usr/bin/env python3
"""
Evaluation script for the CAP model (Single Shared Linear Head).
Loads the trained CAP model and computes classification, explainability, and bias metrics.
"""

import os
import sys
import json
import argparse
import warnings
import random
from itertools import groupby
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize
from tqdm import tqdm
import requests

warnings.filterwarnings('ignore')

# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------
class Config:
    SEED = 42
    MODEL_NAME = 'GroNLP/hateBERT'
    MAX_LENGTH = 128
    BATCH_SIZE = 16
    NUM_LABELS = 3
    LABEL_MAPPING = {'normal': 0, 'offensive': 1, 'hatespeech': 2}
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Split parameters (same as training)
    TRAIN_RATIO = 0.8
    VAL_RATIO = 0.1
    MIN_ANNOTATOR_AGREEMENT = 2

    # Path to the saved CAP model
    BEST_MODEL_PATH = "/content/gdrive/MyDrive/models/CAP_model/best_cap_model.pt"

config = Config()

# Reproducibility

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(config.SEED)


In [ ]:
# Model definition (CAP – shared head)

class CAPModel(nn.Module):
    def __init__(self, model_name, num_labels=3, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.num_labels = num_labels
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_valid_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        subword_states = self.dropout(outputs.last_hidden_state)  # [B, L, H]

        # Shared head -> per-token, per-class logits
        token_logits = self.head(subword_states)                 # [B, L, C]

        # Zero out special/padding token logits
        valid_mask = token_valid_mask.unsqueeze(-1)              # [B, L, 1]
        masked_token_logits = token_logits * valid_mask

        # Mean-pool over valid tokens for classification
        valid_counts = token_valid_mask.sum(dim=1, keepdim=True).clamp(min=1e-9)  # [B, 1]
        sequence_logits = masked_token_logits.sum(dim=1) / valid_counts           # [B, C]

        return sequence_logits, token_logits




In [ ]:
# Dataset (subword-token aligned)

class TokenAlignedDataset(Dataset):
    def __init__(self, words_list, rationales_list, labels, tokenizer, max_length=128):
        self.words_list = words_list
        self.rationales_list = rationales_list
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        words = self.words_list[idx]
        word_rationales = self.rationales_list[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        word_ids = encoding.word_ids(batch_index=0)

        token_valid_mask = torch.zeros(self.max_length, dtype=torch.float)
        token_rationale_mask = torch.zeros(self.max_length, dtype=torch.float)

        for seq_idx, w_id in enumerate(word_ids):
            if w_id is not None:                # ignore [CLS], [SEP], [PAD]
                token_valid_mask[seq_idx] = 1.0
                if w_id < len(word_rationales):
                    token_rationale_mask[seq_idx] = float(word_rationales[w_id])

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long),
            'token_valid_mask': token_valid_mask,
            'token_rationale_mask': token_rationale_mask
        }

def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    labels = torch.stack([item['labels'] for item in batch])
    token_valid_mask = torch.stack([item['token_valid_mask'] for item in batch])
    token_rationale_mask = torch.stack([item['token_rationale_mask'] for item in batch])
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
        'token_valid_mask': token_valid_mask,
        'token_rationale_mask': token_rationale_mask
    }



In [ ]:
# Data loading & split (same as training script)

def load_and_split_data():
    BASE_URL = "https://raw.githubusercontent.com/punyajoy/HateXplain/master/Data/"
    dataset = requests.get(BASE_URL + "dataset.json").json()

    all_post_ids = list(dataset.keys())
    random.seed(config.SEED)
    random.shuffle(all_post_ids)

    total = len(all_post_ids)
    train_end = int(total * config.TRAIN_RATIO)
    val_end = train_end + int(total * config.VAL_RATIO)
    test_ids = all_post_ids[val_end:]

    def load_ids(id_list):
        rows = []
        for tweet_id in id_list:
            info = dataset[tweet_id]
            post_tokens = info["post_tokens"]

            label_counts = Counter(a["label"] for a in info["annotators"])
            if not label_counts:
                continue
            final_label = label_counts.most_common(1)[0][0]
            if final_label not in config.LABEL_MAPPING:
                continue
            if config.MIN_ANNOTATOR_AGREEMENT > 1:
                majority_count = sum(1 for a in info["annotators"] if a["label"] == final_label)
                if majority_count < config.MIN_ANNOTATOR_AGREEMENT:
                    continue

            # Consensus rationale (≥50% of annotators with any rationale)
            consensus_rationale = [0] * len(post_tokens)
            if "rationales" in info and info["rationales"]:
                try:
                    rationale_matrix = np.array(info["rationales"])
                    n_annot = np.sum(np.any(rationale_matrix, axis=1))
                    if n_annot > 0:
                        token_sums = np.sum(rationale_matrix, axis=0)
                        threshold = 0.5 * n_annot
                        consensus_rationale = [1 if s >= threshold else 0 for s in token_sums]
                except ValueError:
                    pass

            # Target communities (for bias metrics)
            targets_all = []
            for ann in info['annotators']:
                if isinstance(ann, dict) and 'target' in ann:
                    t = ann['target']
                    if isinstance(t, list):
                        targets_all.extend(t)
                    elif isinstance(t, str) and t != 'None':
                        targets_all.append(t)
            community_counts = Counter(targets_all)
            final_comms = [c for c, cnt in community_counts.items() if cnt >= 2 and c not in ['None', 'Other']]
            final_target_category = final_comms if final_comms else None

            rows.append({
                "post_id": tweet_id,
                "majority_label": final_label,
                "post_tokens": post_tokens,
                "consensus_rationale": consensus_rationale,
                "final_target_category": final_target_category
            })
        return pd.DataFrame(rows)

    df_test = load_ids(test_ids)

    # Data integrity
    def verify_alignment(df):
        bad = [i for i, row in df.iterrows() if len(row['post_tokens']) != len(row['consensus_rationale'])]
        if bad:
            df = df.drop(index=bad).reset_index(drop=True)
        return df

    df_test = df_test[df_test['post_tokens'].apply(len) > 0].reset_index(drop=True)
    df_test = verify_alignment(df_test)

    test_words = df_test['post_tokens'].tolist()
    test_rats = df_test['consensus_rationale'].tolist()
    test_labels = [config.LABEL_MAPPING[l] for l in df_test['majority_label']]
    test_targets = df_test['final_target_category'].tolist()

    return test_words, test_rats, test_labels, test_targets



In [ ]:
# Prediction helper – extracts per-class token logits and returns
# token probabilities for the predicted class.

def get_predictions(model, dataloader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    all_token_probs = []          # rationale probabilities for the predicted class
    all_token_valid_mask = []
    all_token_rationale_mask = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Predicting"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            token_valid_mask = batch['token_valid_mask'].to(device)
            token_rationale_mask = batch['token_rationale_mask'].to(device)

            sequence_logits, token_logits = model(
                input_ids, attention_mask, token_valid_mask
            )

            probs = torch.softmax(sequence_logits, dim=-1)
            preds = torch.argmax(sequence_logits, dim=-1)

            # Gather the token logits at the predicted class
            B, L, C = token_logits.shape
            pred_idx = preds.view(B, 1, 1).expand(B, L, 1)
            pred_token_logits = token_logits.gather(dim=2, index=pred_idx).squeeze(-1)  # [B, L]
            token_probs = torch.sigmoid(pred_token_logits) * token_valid_mask  # zero out padding

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_token_probs.append(token_probs.cpu())
            all_token_valid_mask.append(token_valid_mask.cpu())
            all_token_rationale_mask.append(token_rationale_mask.cpu())

    return (
        np.array(all_preds),
        np.array(all_labels),
        np.array(all_probs),
        torch.cat(all_token_probs, dim=0),        # probabilities, not raw logits
        torch.cat(all_token_valid_mask, dim=0),
        torch.cat(all_token_rationale_mask, dim=0)
    )



In [ ]:
# Explainability utilities (span-based)

def find_consecutive_spans(binary_mask):
    indices = np.where(binary_mask == 1)[0]
    if len(indices) == 0:
        return []
    spans = []
    for k, g in groupby(enumerate(indices), lambda x: x[1] - x[0]):
        group = list(g)
        spans.append((group[0][1], group[-1][1] + 1))
    return spans

def compute_span_iou_f1(model_spans, human_spans, iou_threshold=0.5):
    if not model_spans and not human_spans:
        return 1.0, 1.0, 1.0
    if not model_spans or not human_spans:
        return 0.0, 0.0, 0.0
    def iou(s1, s2):
        start1, end1 = s1
        start2, end2 = s2
        intersection = max(0, min(end1, end2) - max(start1, start2))
        union = (end1 - start1) + (end2 - start2) - intersection
        return intersection / union if union > 0 else 0.0
    matched_pred = sum(1 for m in model_spans if any(iou(m, h) >= iou_threshold for h in human_spans))
    matched_human = sum(1 for h in human_spans if any(iou(h, m) >= iou_threshold for m in model_spans))
    prec = matched_pred / len(model_spans) if model_spans else 0.0
    rec = matched_human / len(human_spans) if human_spans else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return prec, rec, f1



In [ ]:
# Explainability metrics computation (adapted for CAP)

def compute_explainability_metrics(model, test_loader, device, tokenizer,
                                   preds, labels, probs,
                                   token_probs, token_valid_mask, token_rationale_mask):
    """
    Computes token-level and span-level explainability metrics,
    as well as comprehensiveness and sufficiency, using direct token predictions.
    token_probs are already probabilities (sigmoid of predicted-class token logits).
    """
    print("\n" + "="*80)
    print("Computing Explainability Metrics (CAP model)")
    print("="*80)

    # token_probs, valid_mask, rat_mask are already tensors, we'll convert to numpy
    token_probs_np = token_probs.numpy()
    valid_mask_np = token_valid_mask.numpy()
    rat_mask_np = token_rationale_mask.numpy()

    token_precisions, token_recalls, token_f1s = [], [], []
    iou_scores = []
    span_precs, span_recs, span_f1s = [], [], []
    all_probs_for_auprc, all_human = [], []  # for AUPRC

    toxic_count = 0

    for i in range(len(preds)):
        if labels[i] == 0 or rat_mask_np[i].sum() == 0:
            continue
        toxic_count += 1

        pred_binary = (token_probs_np[i] > 0.5).astype(float) * valid_mask_np[i]
        gold = rat_mask_np[i] * valid_mask_np[i]
        valid_bool = valid_mask_np[i].astype(bool)

        p_mask = pred_binary[valid_bool].astype(bool)
        g_mask = gold[valid_bool].astype(bool)

        tp = (p_mask & g_mask).sum()
        fp = (p_mask & ~g_mask).sum()
        fn = (~p_mask & g_mask).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        token_precisions.append(prec)
        token_recalls.append(rec)
        token_f1s.append(f1)
        intersection = (p_mask & g_mask).sum()
        union = (p_mask | g_mask).sum()
        iou = intersection / union if union > 0 else 0.0
        iou_scores.append(iou)

        # Span-based IoU
        model_spans = find_consecutive_spans(pred_binary)
        human_spans = find_consecutive_spans(gold)
        sp, sr, sf = compute_span_iou_f1(model_spans, human_spans)
        span_precs.append(sp)
        span_recs.append(sr)
        span_f1s.append(sf)

        # AUPRC: collect all token-level probabilities (valid tokens only)
        for j in range(len(valid_bool)):
            if valid_bool[j]:
                all_probs_for_auprc.append(token_probs_np[i, j])
                all_human.append(gold[j])

    # Faithfulness: comprehensiveness & sufficiency
    comprehensiveness_scores, sufficiency_scores = [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Faithfulness"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_batch = batch['labels'].to(device)
            token_valid_mask_batch = batch['token_valid_mask'].to(device)
            token_rationale_mask_batch = batch['token_rationale_mask'].to(device)

            # Recompute predictions for this batch
            seq_logits, token_logits = model(input_ids, attention_mask, token_valid_mask_batch)
            probs_batch = torch.softmax(seq_logits, dim=-1)
            preds_batch = torch.argmax(seq_logits, dim=-1)

            # Get token probabilities for predicted class
            B, L, C = token_logits.shape
            pred_idx_batch = preds_batch.view(B, 1, 1).expand(B, L, 1)
            pred_token_logits_batch = token_logits.gather(dim=2, index=pred_idx_batch).squeeze(-1)
            tok_probs_batch = torch.sigmoid(pred_token_logits_batch) * token_valid_mask_batch

            for i in range(len(labels_batch)):
                if labels_batch[i].item() == 0 or token_rationale_mask_batch[i].sum() == 0:
                    continue
                orig_prob = probs_batch[i, labels_batch[i]].item()
                pred_tok_binary = (tok_probs_batch[i] > 0.5).float() * token_valid_mask_batch[i]
                M_set = set(torch.where(pred_tok_binary == 1)[0].tolist())

                # Comprehensiveness: mask those tokens
                masked_ids = input_ids[i].clone()
                for t_idx in M_set:
                    if t_idx < len(masked_ids):
                        masked_ids[t_idx] = tokenizer.mask_token_id
                masked_out = model(masked_ids.unsqueeze(0),
                                   attention_mask[i].unsqueeze(0),
                                   token_valid_mask_batch[i].unsqueeze(0))
                masked_prob = torch.softmax(masked_out[0], dim=-1)[0, labels_batch[i]].item()
                comprehensiveness_scores.append(orig_prob - masked_prob)

                # Sufficiency: keep only those tokens
                suff_ids = torch.full_like(input_ids[i], tokenizer.pad_token_id)
                suff_att = torch.zeros_like(attention_mask[i])
                suff_ids[0] = tokenizer.cls_token_id
                suff_att[0] = 1
                last_idx = 0
                for t_idx in M_set:
                    if t_idx < len(suff_ids):
                        suff_ids[t_idx] = input_ids[i][t_idx]
                        suff_att[t_idx] = 1
                        if t_idx > last_idx:
                            last_idx = t_idx
                if last_idx + 1 < len(suff_ids):
                    suff_ids[last_idx + 1] = tokenizer.sep_token_id
                    suff_att[last_idx + 1] = 1

                # Create a new token_valid_mask for sufficiency: only CLS, kept tokens, SEP
                suff_token_valid = torch.zeros_like(token_valid_mask_batch[i])
                suff_token_valid[0] = 1.0
                for t_idx in M_set:
                    if t_idx < len(suff_token_valid):
                        suff_token_valid[t_idx] = 1.0
                if last_idx + 1 < len(suff_token_valid):
                    suff_token_valid[last_idx + 1] = 1.0

                suff_out = model(suff_ids.unsqueeze(0), suff_att.unsqueeze(0), suff_token_valid.unsqueeze(0))
                suff_prob = torch.softmax(suff_out[0], dim=-1)[0, labels_batch[i]].item()
                sufficiency_scores.append(orig_prob - suff_prob)

    if toxic_count == 0:
        print("No toxic examples with rationales found.")
        return None

    auprc = average_precision_score(all_human, all_probs_for_auprc) if all_human else 0.0

    results = {
        'plausibility': {
            'token_precision': np.mean(token_precisions) if token_precisions else 0.0,
            'token_recall': np.mean(token_recalls) if token_recalls else 0.0,
            'token_f1': np.mean(token_f1s) if token_f1s else 0.0,
            'iou': np.mean(iou_scores) if iou_scores else 0.0,
            'span_iou_precision': np.mean(span_precs) if span_precs else 0.0,
            'span_iou_recall': np.mean(span_recs) if span_recs else 0.0,
            'span_iou_f1': np.mean(span_f1s) if span_f1s else 0.0,
            'auprc': auprc,
        },
        'faithfulness': {
            'comprehensiveness': np.mean(comprehensiveness_scores) if comprehensiveness_scores else 0.0,
            'sufficiency': np.mean(sufficiency_scores) if sufficiency_scores else 0.0,
        },
        'num_examples': toxic_count
    }

    print(f"\nExamples evaluated: {toxic_count}")
    print("\nPlausibility:")
    print(f"  Token F1:          {results['plausibility']['token_f1']:.3f}")
    print(f"  Token IOU:         {results['plausibility']['iou']:.3f}")
    print(f"  Span IOU F1:       {results['plausibility']['span_iou_f1']:.3f}")
    print(f"  AUPRC:             {results['plausibility']['auprc']:.3f}")
    print("\nFaithfulness:")
    print(f"  Comprehensiveness: {results['faithfulness']['comprehensiveness']:.3f}")
    print(f"  Sufficiency:       {results['faithfulness']['sufficiency']:.3f}")
    return results



In [ ]:
# Bias metrics (same logic, adapted for token_valid_mask)

def compute_bias_metrics(model, test_loader, device, target_categories):
    print("\n" + "="*80)
    print("Computing Bias Metrics (HateXplain Method)")
    print("="*80)
    model.eval()
    selected = ['African', 'Islam', 'Jewish', 'Homosexual', 'Women',
                'Refugee', 'Arab', 'Caucasian', 'Asian', 'Hispanic']

    all_labels, all_probs, all_targets = [], [], []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(test_loader, desc="Bias evaluation")):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            token_valid_mask = batch['token_valid_mask'].to(device)

            # Bias only needs classification probabilities
            sequence_logits, _ = model(input_ids, attention_mask, token_valid_mask)
            probs = torch.softmax(sequence_logits, dim=-1)
            toxic_probs = probs[:, 1:].sum(dim=1).cpu().numpy()

            for i in range(len(labels)):
                # index in target_categories: batch_idx * batch_size + i
                data_idx = batch_idx * test_loader.batch_size + i
                all_labels.append(1 if labels[i].item() > 0 else 0)
                all_probs.append(toxic_probs[i])
                all_targets.append(target_categories[data_idx])

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    community_metrics = {}
    for comm in selected:
        mask = np.array([comm in (t or []) for t in all_targets])
        if mask.sum() == 0:
            continue
        sub_labels = all_labels[mask]
        sub_probs = all_probs[mask]
        subgroup_auc = roc_auc_score(sub_labels, sub_probs) if len(np.unique(sub_labels)) > 1 else 0.5

        bpsn_mask = (mask & (all_labels == 0)) | (~mask & (all_labels == 1))
        bpsn_labels = all_labels[bpsn_mask]
        bpsn_probs = all_probs[bpsn_mask]
        bpsn_auc = roc_auc_score(bpsn_labels, bpsn_probs) if bpsn_mask.sum() > 0 and len(np.unique(bpsn_labels)) > 1 else 0.5

        bnsp_mask = (mask & (all_labels == 1)) | (~mask & (all_labels == 0))
        bnsp_labels = all_labels[bnsp_mask]
        bnsp_probs = all_probs[bnsp_mask]
        bnsp_auc = roc_auc_score(bnsp_labels, bnsp_probs) if bnsp_mask.sum() > 0 and len(np.unique(bnsp_labels)) > 1 else 0.5

        community_metrics[comm] = {
            'mentions': int(mask.sum()),
            'subgroup_auc': float(subgroup_auc),
            'bpsn_auc': float(bpsn_auc),
            'bnsp_auc': float(bnsp_auc),
        }

    p = -5
    valid = [c for c in community_metrics if community_metrics[c]['mentions'] > 0]
    if valid:
        gmb_sub = np.power(np.mean(np.power([community_metrics[c]['subgroup_auc'] for c in valid], p)), 1/p)
        gmb_bpsn = np.power(np.mean(np.power([community_metrics[c]['bpsn_auc'] for c in valid], p)), 1/p)
        gmb_bnsp = np.power(np.mean(np.power([community_metrics[c]['bnsp_auc'] for c in valid], p)), 1/p)
    else:
        gmb_sub = gmb_bpsn = gmb_bnsp = 0.5

    overall_auc = roc_auc_score(all_labels, all_probs)

    results = {
        'overall_auc': float(overall_auc),
        'communities': community_metrics,
        'gmb_metrics': {
            'gmb_subgroup_auc': float(gmb_sub),
            'gmb_bpsn_auc': float(gmb_bpsn),
            'gmb_bnsp_auc': float(gmb_bnsp),
        },
        'n_communities_analyzed': len(valid),
    }
    print(f"\nOverall AUC: {overall_auc:.4f}")
    for comm, m in sorted(community_metrics.items(), key=lambda x: x[1]['mentions'], reverse=True):
        print(f"{comm:<15} {m['mentions']:<8} {m['subgroup_auc']:<12.4f} {m['bpsn_auc']:<12.4f} {m['bnsp_auc']:<12.4f}")
    print(f"\nGMB (p={p}): Subgroup={gmb_sub:.4f}  BPSN={gmb_bpsn:.4f}  BNSP={gmb_bnsp:.4f}")
    return results

# Error analysis

def save_error_cases(predictions, labels, probabilities, texts, output_file="error_cases.json"):
    label_names = ['Normal', 'Offensive', 'Hate speech']
    error_cases = []
    for idx in range(len(predictions)):
        pred, true = int(predictions[idx]), int(labels[idx])
        if pred != true:
            prob = [float(x) for x in probabilities[idx]]
            error_cases.append({
                "text": texts[idx],
                "true_label": label_names[true],
                "predicted_label": label_names[pred],
                "confidence": float(max(prob)),
                "predicted_probs": {"normal": prob[0], "offensive": prob[1], "hate_speech": prob[2]},
                "error_type": f"{label_names[true]}_as_{label_names[pred]}"
            })
    with open(output_file, "w") as f:
        json.dump(error_cases, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(error_cases)} error cases to {output_file}")



In [ ]:
# Main evaluation

def main():
    parser = argparse.ArgumentParser(description="Evaluate CAP Model")
    parser.add_argument('--model_path', type=str, default=config.BEST_MODEL_PATH,
                        help="Path to the trained CAP model checkpoint")
    parser.add_argument('--batch_size', type=int, default=config.BATCH_SIZE)
    parser.add_argument('--save_results', type=str, default=None)
    parser.add_argument('--no_explainability', action='store_true')
    parser.add_argument('--no_bias', action='store_true')
    args, unknown = parser.parse_known_args()

    device = config.DEVICE
    print(f"Using device: {device}")

    # Load test data
    print("Loading and preparing test split...")
    test_words, test_rats, test_labels, test_targets = load_and_split_data()
    print(f"Test samples: {len(test_words)}")

    tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
    test_dataset = TokenAlignedDataset(test_words, test_rats, test_labels, tokenizer, max_length=config.MAX_LENGTH)
    test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate_fn)

    # Load model
    print(f"Loading model from {args.model_path}")
    model = CAPModel(config.MODEL_NAME, num_labels=config.NUM_LABELS)
    state_dict = torch.load(args.model_path, map_location='cpu')
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    # Get predictions and token probabilities
    print("\nRunning classification evaluation...")
    preds, labels, probs, token_probs, token_valid, token_rats = get_predictions(
        model, test_loader, device
    )

    # Classification metrics
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    try:
        labels_bin = label_binarize(labels, classes=[0,1,2])
        auroc = roc_auc_score(labels_bin, probs, average='macro', multi_class='ovr')
    except:
        auroc = 0.5

    print("\nTest Set Performance:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Macro F1:  {macro_f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  AUROC:     {auroc:.4f}")
    print("\n", classification_report(labels, preds, target_names=['Normal', 'Offensive', 'Hate speech'], digits=4))

    # Error cases
    texts = [' '.join(tokens) for tokens in test_words]
    save_error_cases(preds, labels, probs, texts)

    # Explainability
    explainability_results = None
    if not args.no_explainability:
        explainability_results = compute_explainability_metrics(
            model, test_loader, device, tokenizer,
            preds, labels, probs,
            token_probs, token_valid, token_rats
        )

    # Bias metrics
    bias_results = None
    if not args.no_bias and any(t is not None for t in test_targets):
        bias_results = compute_bias_metrics(model, test_loader, device, test_targets)
    elif not args.no_bias:
        print("\nSkipping bias metrics: no target categories found.")

    # Summary
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"F1: {macro_f1:.4f}  |  Accuracy: {acc:.4f}  |  AUROC: {auroc:.4f}")
    if explainability_results:
        p = explainability_results['plausibility']
        f = explainability_results['faithfulness']
        print(f"Plausibility: Token F1={p['token_f1']:.3f}  Span IOU F1={p['span_iou_f1']:.3f}  AUPRC={p['auprc']:.3f}")
        print(f"Faithfulness: Comp={f['comprehensiveness']:.3f}  Suff={f['sufficiency']:.3f}")
    if bias_results:
        g = bias_results['gmb_metrics']
        print(f"Bias: Overall AUC={bias_results['overall_auc']:.4f}  GMB Sub={g['gmb_subgroup_auc']:.4f}  "
              f"BPSN={g['gmb_bpsn_auc']:.4f}  BNSP={g['gmb_bnsp_auc']:.4f}")

    if args.save_results:
        save_data = {
            'accuracy': acc,
            'macro_f1': macro_f1,
            'precision': precision,
            'recall': recall,
            'auroc': auroc,
            'predictions': preds.tolist(),
            'labels': labels.tolist(),
            'probabilities': probs.tolist(),
        }
        if explainability_results:
            save_data['explainability'] = explainability_results
        if bias_results:
            save_data['bias'] = bias_results
        with open(args.save_results, 'w') as f:
            json.dump(save_data, f, indent=2)
        print(f"Results saved to {args.save_results}")

if __name__ == "__main__":
    main()

Using device: cuda
Loading and preparing test split...
Test samples: 1923
Loading model from /content/gdrive/MyDrive/models/CAP_model/best_cap_model.pt


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Running classification evaluation...


Predicting: 100%|██████████| 121/121 [00:13<00:00,  8.80it/s]



Test Set Performance:
  Accuracy:  0.6937
  Macro F1:  0.6842
  Precision: 0.6828
  Recall:    0.6869
  AUROC:     0.8554

               precision    recall  f1-score   support

      Normal     0.7782    0.7238    0.7500       800
   Offensive     0.5304    0.5460    0.5380       544
 Hate speech     0.7399    0.7910    0.7646       579

    accuracy                         0.6937      1923
   macro avg     0.6828    0.6869    0.6842      1923
weighted avg     0.6966    0.6937    0.6944      1923

Saved 589 error cases to error_cases.json

Computing Explainability Metrics (CAP model)


Faithfulness: 100%|██████████| 121/121 [00:38<00:00,  3.17it/s]



Examples evaluated: 1122

Plausibility:
  Token F1:          0.695
  Token IOU:         0.599
  Span IOU F1:       0.570
  AUPRC:             0.753

Faithfulness:
  Comprehensiveness: 0.507
  Sufficiency:       -0.026

Computing Bias Metrics (HateXplain Method)


Bias evaluation: 100%|██████████| 121/121 [00:13<00:00,  8.69it/s]



Overall AUC: 0.8768
African         310      0.8815       0.7941       0.9203      
Islam           197      0.8192       0.7720       0.9022      
Jewish          185      0.9066       0.7488       0.9553      
Homosexual      177      0.8788       0.8457       0.9011      
Women           155      0.7354       0.8285       0.8301      
Refugee         81       0.7816       0.8961       0.7763      
Arab            64       0.5000       0.5000       0.9336      
Caucasian       43       0.8835       0.9614       0.7366      
Asian           39       0.8667       0.9142       0.8170      
Hispanic        27       1.0000       0.9745       0.9511      

GMB (p=-5): Subgroup=0.7161  BPSN=0.7148  BNSP=0.8521

SUMMARY
F1: 0.6842  |  Accuracy: 0.6937  |  AUROC: 0.8554
Plausibility: Token F1=0.695  Span IOU F1=0.570  AUPRC=0.753
Faithfulness: Comp=0.507  Suff=-0.026
Bias: Overall AUC=0.8768  GMB Sub=0.7161  BPSN=0.7148  BNSP=0.8521


lambda = 10

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import os
import random
import csv
import requests
from collections import Counter

# Config & Reproducibility

SEED = 42
MODEL_NAME = 'GroNLP/hateBERT'
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
LAMBDA_RAT = 10
SAVE_DIR = "/content/gdrive/MyDrive/models/CAP_model(lambda=10)"
LOG_PATH = os.path.join(SAVE_DIR, "training_log.csv")

LABEL2ID = {"normal": 0, "offensive": 1, "hatespeech": 2}
NUM_LABELS = len(LABEL2ID)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
# Setup & Training Loop

print(f"Loading {MODEL_NAME} tokenizer and initializing CAP model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = CAPModel(MODEL_NAME, num_labels=NUM_LABELS).to(device)

train_dataset = TokenAlignedDataset(train_words, train_rats, train_labels, tokenizer)
val_dataset   = TokenAlignedDataset(val_words, val_rats, val_labels, tokenizer)
test_dataset  = TokenAlignedDataset(test_words, test_rats, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

cls_loss_fn = nn.CrossEntropyLoss(weight=class_weights)

with open(LOG_PATH, 'w', newline='') as f:
    csv.writer(f).writerow(['epoch', 'train_loss', 'cls_loss', 'rat_loss', 'val_acc', 'val_macro_f1', 'val_token_f1', 'val_iou'])

best_val_f1 = -1.0
best_model_path = os.path.join(SAVE_DIR, "best_cap_model.pt")

print(f"Starting CAP training on {device}...")

for epoch in range(EPOCHS):
    model.train()
    total_loss, total_cls_loss, total_rat_loss = 0.0, 0.0, 0.0
    n_valid_batches = 0

    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        token_valid_mask = batch['token_valid_mask'].to(device)
        token_rationale_mask = batch['token_rationale_mask'].to(device)

        optimizer.zero_grad()

        sequence_logits, token_logits = model(input_ids, attention_mask, token_valid_mask)

        cls_loss = cls_loss_fn(sequence_logits, labels)
        rat_loss = token_rationale_loss(token_logits, labels, token_rationale_mask, token_valid_mask)
        loss = cls_loss + (LAMBDA_RAT * rat_loss)

        if not torch.isfinite(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        total_cls_loss += cls_loss.item()
        total_rat_loss += rat_loss.item()
        n_valid_batches += 1

        loop.set_description(f'Epoch {epoch+1}')
        loop.set_postfix(Loss=loss.item(), Cls=cls_loss.item(), RatBCE=rat_loss.item())

    denom = max(n_valid_batches, 1)
    avg_loss = total_loss / denom
    avg_cls = total_cls_loss / denom
    avg_rat = total_rat_loss / denom

    val_acc, val_macro_f1, val_token_f1, val_iou, _, _ = evaluate(model, val_loader)

    print(f"\nEpoch {epoch+1} | Loss: {avg_loss:.4f} | Cls: {avg_cls:.4f} | RatBCE: {avg_rat:.4f}")
    print(f"Validation -> Acc: {val_acc:.4f} | Macro-F1: {val_macro_f1:.4f} | Token-F1: {val_token_f1:.4f} | IOU: {val_iou:.4f}\n")

    with open(LOG_PATH, 'a', newline='') as f:
        csv.writer(f).writerow([epoch+1, avg_loss, avg_cls, avg_rat, val_acc, val_macro_f1, val_token_f1, val_iou])

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"New best model saved based on Macro-F1: {val_macro_f1:.4f}")

print("Training complete!")

# Evaluation

model.load_state_dict(torch.load(best_model_path))
test_acc, test_macro_f1, test_token_f1, test_iou, test_preds, test_labels_out = evaluate(model, test_loader)

print(f"\n" + "="*60)
print(f"FINAL TEST SET RESULTS (CAP: Single Shared Linear Head)")
print(f"="*60)
print(f"Sentence Accuracy      : {test_acc:.4f}")
print(f"Sentence Macro-F1      : {test_macro_f1:.4f}")
print(f"Subword-Token F1       : {test_token_f1:.4f}")
print(f"Subword-Token IOU      : {test_iou:.4f}")
print(f"="*60)
print(classification_report(test_labels_out, test_preds, target_names=list(LABEL2ID.keys()), digits=4))

Loading GroNLP/hateBERT tokenizer and initializing CAP model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting CAP training on cuda...


Epoch 1: 100%|██████████| 962/962 [05:55<00:00,  2.71it/s, Cls=1, Loss=6.85, RatBCE=0.584]



Epoch 1 | Loss: 6.1131 | Cls: 0.9752 | RatBCE: 0.5138
Validation -> Acc: 0.6636 | Macro-F1: 0.6352 | Token-F1: 0.6694 | IOU: 0.5793

New best model saved based on Macro-F1: 0.6352


Epoch 2: 100%|██████████| 962/962 [05:50<00:00,  2.74it/s, Cls=0.355, Loss=0.355, RatBCE=0]



Epoch 2 | Loss: 4.7735 | Cls: 0.7812 | RatBCE: 0.3992
Validation -> Acc: 0.6807 | Macro-F1: 0.6719 | Token-F1: 0.7380 | IOU: 0.6470

New best model saved based on Macro-F1: 0.6719


Epoch 3: 100%|██████████| 962/962 [05:51<00:00,  2.73it/s, Cls=1.1, Loss=9.19, RatBCE=0.809]



Epoch 3 | Loss: 3.9712 | Cls: 0.7247 | RatBCE: 0.3246
Validation -> Acc: 0.6801 | Macro-F1: 0.6756 | Token-F1: 0.7418 | IOU: 0.6472

New best model saved based on Macro-F1: 0.6756
Training complete!

FINAL TEST SET RESULTS (CAP: Single Shared Linear Head)
Sentence Accuracy      : 0.6739
Sentence Macro-F1      : 0.6653
Subword-Token F1       : 0.7354
Subword-Token IOU      : 0.6416
              precision    recall  f1-score   support

      normal     0.7623    0.6975    0.7285       800
   offensive     0.5123    0.5368    0.5242       544
  hatespeech     0.7182    0.7703    0.7433       579

    accuracy                         0.6739      1923
   macro avg     0.6643    0.6682    0.6653      1923
weighted avg     0.6783    0.6739    0.6752      1923



In [ ]:
#!/usr/bin/env python3
"""
Evaluation script for the CAP model (Single Shared Linear Head).
Loads the trained CAP model and computes classification, explainability, and bias metrics.
"""

import os
import sys
import json
import argparse
import warnings
import random
from itertools import groupby
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize
from tqdm import tqdm
import requests

warnings.filterwarnings('ignore')

# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------
class Config:
    SEED = 42
    MODEL_NAME = 'GroNLP/hateBERT'
    MAX_LENGTH = 128
    BATCH_SIZE = 16
    NUM_LABELS = 3
    LABEL_MAPPING = {'normal': 0, 'offensive': 1, 'hatespeech': 2}
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Split parameters (same as training)
    TRAIN_RATIO = 0.8
    VAL_RATIO = 0.1
    MIN_ANNOTATOR_AGREEMENT = 2

    # Path to the saved CAP model
    BEST_MODEL_PATH = "/content/gdrive/MyDrive/models/CAP_model(lambda=10)/best_cap_model.pt"

config = Config()

# Reproducibility

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(config.SEED)

In [ ]:
# Main evaluation

def main():
    parser = argparse.ArgumentParser(description="Evaluate CAP Model")
    parser.add_argument('--model_path', type=str, default=config.BEST_MODEL_PATH,
                        help="Path to the trained CAP model checkpoint")
    parser.add_argument('--batch_size', type=int, default=config.BATCH_SIZE)
    parser.add_argument('--save_results', type=str, default=None)
    parser.add_argument('--no_explainability', action='store_true')
    parser.add_argument('--no_bias', action='store_true')
    args, unknown = parser.parse_known_args()

    device = config.DEVICE
    print(f"Using device: {device}")

    # Load test data
    print("Loading and preparing test split...")
    test_words, test_rats, test_labels, test_targets = load_and_split_data()
    print(f"Test samples: {len(test_words)}")

    tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
    test_dataset = TokenAlignedDataset(test_words, test_rats, test_labels, tokenizer, max_length=config.MAX_LENGTH)
    test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate_fn)

    # Load model
    print(f"Loading model from {args.model_path}")
    model = CAPModel(config.MODEL_NAME, num_labels=config.NUM_LABELS)
    state_dict = torch.load(args.model_path, map_location='cpu')
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    # Get predictions and token probabilities
    print("\nRunning classification evaluation...")
    preds, labels, probs, token_probs, token_valid, token_rats = get_predictions(
        model, test_loader, device
    )

    # Classification metrics
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    try:
        labels_bin = label_binarize(labels, classes=[0,1,2])
        auroc = roc_auc_score(labels_bin, probs, average='macro', multi_class='ovr')
    except:
        auroc = 0.5

    print("\nTest Set Performance:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Macro F1:  {macro_f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  AUROC:     {auroc:.4f}")
    print("\n", classification_report(labels, preds, target_names=['Normal', 'Offensive', 'Hate speech'], digits=4))

    # Error cases
    texts = [' '.join(tokens) for tokens in test_words]
    save_error_cases(preds, labels, probs, texts)

    # Explainability
    explainability_results = None
    if not args.no_explainability:
        explainability_results = compute_explainability_metrics(
            model, test_loader, device, tokenizer,
            preds, labels, probs,
            token_probs, token_valid, token_rats
        )

    # Bias metrics
    bias_results = None
    if not args.no_bias and any(t is not None for t in test_targets):
        bias_results = compute_bias_metrics(model, test_loader, device, test_targets)
    elif not args.no_bias:
        print("\nSkipping bias metrics: no target categories found.")

    # Summary
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"F1: {macro_f1:.4f}  |  Accuracy: {acc:.4f}  |  AUROC: {auroc:.4f}")
    if explainability_results:
        p = explainability_results['plausibility']
        f = explainability_results['faithfulness']
        print(f"Plausibility: Token F1={p['token_f1']:.3f}  Span IOU F1={p['span_iou_f1']:.3f}  AUPRC={p['auprc']:.3f}")
        print(f"Faithfulness: Comp={f['comprehensiveness']:.3f}  Suff={f['sufficiency']:.3f}")
    if bias_results:
        g = bias_results['gmb_metrics']
        print(f"Bias: Overall AUC={bias_results['overall_auc']:.4f}  GMB Sub={g['gmb_subgroup_auc']:.4f}  "
              f"BPSN={g['gmb_bpsn_auc']:.4f}  BNSP={g['gmb_bnsp_auc']:.4f}")

    if args.save_results:
        save_data = {
            'accuracy': acc,
            'macro_f1': macro_f1,
            'precision': precision,
            'recall': recall,
            'auroc': auroc,
            'predictions': preds.tolist(),
            'labels': labels.tolist(),
            'probabilities': probs.tolist(),
        }
        if explainability_results:
            save_data['explainability'] = explainability_results
        if bias_results:
            save_data['bias'] = bias_results
        with open(args.save_results, 'w') as f:
            json.dump(save_data, f, indent=2)
        print(f"Results saved to {args.save_results}")

if __name__ == "__main__":
    main()

Using device: cuda
Loading and preparing test split...
Test samples: 1923
Loading model from /content/gdrive/MyDrive/models/CAP_model(lambda=10)/best_cap_model.pt


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Running classification evaluation...


Predicting: 100%|██████████| 121/121 [00:15<00:00,  7.63it/s]



Test Set Performance:
  Accuracy:  0.6739
  Macro F1:  0.6653
  Precision: 0.6643
  Recall:    0.6682
  AUROC:     0.8409

               precision    recall  f1-score   support

      Normal     0.7623    0.6975    0.7285       800
   Offensive     0.5123    0.5368    0.5242       544
 Hate speech     0.7182    0.7703    0.7433       579

    accuracy                         0.6739      1923
   macro avg     0.6643    0.6682    0.6653      1923
weighted avg     0.6783    0.6739    0.6752      1923

Saved 627 error cases to error_cases.json

Computing Explainability Metrics (CAP model)


Faithfulness: 100%|██████████| 121/121 [00:41<00:00,  2.94it/s]



Examples evaluated: 1122

Plausibility:
  Token F1:          0.735
  Token IOU:         0.642
  Span IOU F1:       0.609
  AUPRC:             0.786

Faithfulness:
  Comprehensiveness: 0.419
  Sufficiency:       -0.064

Computing Bias Metrics (HateXplain Method)


Bias evaluation: 100%|██████████| 121/121 [00:14<00:00,  8.45it/s]



Overall AUC: 0.8603
African         310      0.8481       0.7543       0.9176      
Islam           197      0.8541       0.8137       0.8875      
Jewish          185      0.8858       0.7284       0.9329      
Homosexual      177      0.8709       0.8570       0.8743      
Women           155      0.7203       0.8094       0.7987      
Refugee         81       0.6995       0.8973       0.6937      
Arab            64       0.5000       0.5000       0.9365      
Caucasian       43       0.8665       0.9519       0.7070      
Asian           39       0.9296       0.9385       0.7935      
Hispanic        27       1.0000       0.9613       0.9396      

GMB (p=-5): Subgroup=0.7087  BPSN=0.7126  BNSP=0.8161

SUMMARY
F1: 0.6653  |  Accuracy: 0.6739  |  AUROC: 0.8409
Plausibility: Token F1=0.735  Span IOU F1=0.609  AUPRC=0.786
Faithfulness: Comp=0.419  Suff=-0.064
Bias: Overall AUC=0.8603  GMB Sub=0.7087  BPSN=0.7126  BNSP=0.8161


lambda = 5

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import os
import random
import csv
import requests
from collections import Counter

# Config & Reproducibility

SEED = 42
MODEL_NAME = 'GroNLP/hateBERT'
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
LAMBDA_RAT = 5
SAVE_DIR = "/content/gdrive/MyDrive/models/CAP_model(lambda=5)"
LOG_PATH = os.path.join(SAVE_DIR, "training_log.csv")

LABEL2ID = {"normal": 0, "offensive": 1, "hatespeech": 2}
NUM_LABELS = len(LABEL2ID)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
# Setup & Training Loop

print(f"Loading {MODEL_NAME} tokenizer and initializing CAP model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = CAPModel(MODEL_NAME, num_labels=NUM_LABELS).to(device)

train_dataset = TokenAlignedDataset(train_words, train_rats, train_labels, tokenizer)
val_dataset   = TokenAlignedDataset(val_words, val_rats, val_labels, tokenizer)
test_dataset  = TokenAlignedDataset(test_words, test_rats, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

cls_loss_fn = nn.CrossEntropyLoss(weight=class_weights)

with open(LOG_PATH, 'w', newline='') as f:
    csv.writer(f).writerow(['epoch', 'train_loss', 'cls_loss', 'rat_loss', 'val_acc', 'val_macro_f1', 'val_token_f1', 'val_iou'])

best_val_f1 = -1.0
best_model_path = os.path.join(SAVE_DIR, "best_cap_model.pt")

print(f"Starting CAP training on {device}...")

for epoch in range(EPOCHS):
    model.train()
    total_loss, total_cls_loss, total_rat_loss = 0.0, 0.0, 0.0
    n_valid_batches = 0

    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        token_valid_mask = batch['token_valid_mask'].to(device)
        token_rationale_mask = batch['token_rationale_mask'].to(device)

        optimizer.zero_grad()

        sequence_logits, token_logits = model(input_ids, attention_mask, token_valid_mask)

        cls_loss = cls_loss_fn(sequence_logits, labels)
        rat_loss = token_rationale_loss(token_logits, labels, token_rationale_mask, token_valid_mask)
        loss = cls_loss + (LAMBDA_RAT * rat_loss)

        if not torch.isfinite(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        total_cls_loss += cls_loss.item()
        total_rat_loss += rat_loss.item()
        n_valid_batches += 1

        loop.set_description(f'Epoch {epoch+1}')
        loop.set_postfix(Loss=loss.item(), Cls=cls_loss.item(), RatBCE=rat_loss.item())

    denom = max(n_valid_batches, 1)
    avg_loss = total_loss / denom
    avg_cls = total_cls_loss / denom
    avg_rat = total_rat_loss / denom

    val_acc, val_macro_f1, val_token_f1, val_iou, _, _ = evaluate(model, val_loader)

    print(f"\nEpoch {epoch+1} | Loss: {avg_loss:.4f} | Cls: {avg_cls:.4f} | RatBCE: {avg_rat:.4f}")
    print(f"Validation -> Acc: {val_acc:.4f} | Macro-F1: {val_macro_f1:.4f} | Token-F1: {val_token_f1:.4f} | IOU: {val_iou:.4f}\n")

    with open(LOG_PATH, 'a', newline='') as f:
        csv.writer(f).writerow([epoch+1, avg_loss, avg_cls, avg_rat, val_acc, val_macro_f1, val_token_f1, val_iou])

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"New best model saved based on Macro-F1: {val_macro_f1:.4f}")

print("Training complete!")

# Evaluation

model.load_state_dict(torch.load(best_model_path))
test_acc, test_macro_f1, test_token_f1, test_iou, test_preds, test_labels_out = evaluate(model, test_loader)

print(f"\n" + "="*60)
print(f"FINAL TEST SET RESULTS (CAP: Single Shared Linear Head)")
print(f"="*60)
print(f"Sentence Accuracy      : {test_acc:.4f}")
print(f"Sentence Macro-F1      : {test_macro_f1:.4f}")
print(f"Subword-Token F1       : {test_token_f1:.4f}")
print(f"Subword-Token IOU      : {test_iou:.4f}")
print(f"="*60)
print(classification_report(test_labels_out, test_preds, target_names=list(LABEL2ID.keys()), digits=4))

Loading GroNLP/hateBERT tokenizer and initializing CAP model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting CAP training on cuda...


Epoch 1: 100%|██████████| 962/962 [05:55<00:00,  2.70it/s, Cls=0.937, Loss=3.91, RatBCE=0.595]



Epoch 1 | Loss: 3.4867 | Cls: 0.9119 | RatBCE: 0.5150
Validation -> Acc: 0.6833 | Macro-F1: 0.6583 | Token-F1: 0.6870 | IOU: 0.5954

New best model saved based on Macro-F1: 0.6583


Epoch 2: 100%|██████████| 962/962 [05:50<00:00,  2.74it/s, Cls=0.412, Loss=0.412, RatBCE=0]



Epoch 2 | Loss: 2.7301 | Cls: 0.7164 | RatBCE: 0.4028
Validation -> Acc: 0.6926 | Macro-F1: 0.6862 | Token-F1: 0.7440 | IOU: 0.6510

New best model saved based on Macro-F1: 0.6862


Epoch 3: 100%|██████████| 962/962 [05:51<00:00,  2.74it/s, Cls=1.09, Loss=5.05, RatBCE=0.792]



Epoch 3 | Loss: 2.3119 | Cls: 0.6534 | RatBCE: 0.3317
Validation -> Acc: 0.6895 | Macro-F1: 0.6850 | Token-F1: 0.7483 | IOU: 0.6541

Training complete!

FINAL TEST SET RESULTS (CAP: Single Shared Linear Head)
Sentence Accuracy      : 0.6916
Sentence Macro-F1      : 0.6811
Subword-Token F1       : 0.7225
Subword-Token IOU      : 0.6283
              precision    recall  f1-score   support

      normal     0.7531    0.7475    0.7503       800
   offensive     0.5348    0.5368    0.5358       544
  hatespeech     0.7547    0.7599    0.7573       579

    accuracy                         0.6916      1923
   macro avg     0.6809    0.6814    0.6811      1923
weighted avg     0.6919    0.6916    0.6917      1923



In [ ]:
#!/usr/bin/env python3
"""
Evaluation script for the CAP model (Single Shared Linear Head).
Loads the trained CAP model and computes classification, explainability, and bias metrics.
"""

import os
import sys
import json
import argparse
import warnings
import random
from itertools import groupby
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize
from tqdm import tqdm
import requests

warnings.filterwarnings('ignore')

# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------
class Config:
    SEED = 42
    MODEL_NAME = 'GroNLP/hateBERT'
    MAX_LENGTH = 128
    BATCH_SIZE = 16
    NUM_LABELS = 3
    LABEL_MAPPING = {'normal': 0, 'offensive': 1, 'hatespeech': 2}
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Split parameters (same as training)
    TRAIN_RATIO = 0.8
    VAL_RATIO = 0.1
    MIN_ANNOTATOR_AGREEMENT = 2

    # Path to the saved CAP model
    BEST_MODEL_PATH = "/content/gdrive/MyDrive/models/CAP_model(lambda=5)/best_cap_model.pt"

config = Config()

# Reproducibility

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(config.SEED)

In [ ]:
# Main evaluation

def main():
    parser = argparse.ArgumentParser(description="Evaluate CAP Model")
    parser.add_argument('--model_path', type=str, default=config.BEST_MODEL_PATH,
                        help="Path to the trained CAP model checkpoint")
    parser.add_argument('--batch_size', type=int, default=config.BATCH_SIZE)
    parser.add_argument('--save_results', type=str, default=None)
    parser.add_argument('--no_explainability', action='store_true')
    parser.add_argument('--no_bias', action='store_true')
    args, unknown = parser.parse_known_args()

    device = config.DEVICE
    print(f"Using device: {device}")

    # Load test data
    print("Loading and preparing test split...")
    test_words, test_rats, test_labels, test_targets = load_and_split_data()
    print(f"Test samples: {len(test_words)}")

    tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
    test_dataset = TokenAlignedDataset(test_words, test_rats, test_labels, tokenizer, max_length=config.MAX_LENGTH)
    test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate_fn)

    # Load model
    print(f"Loading model from {args.model_path}")
    model = CAPModel(config.MODEL_NAME, num_labels=config.NUM_LABELS)
    state_dict = torch.load(args.model_path, map_location='cpu')
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    # Get predictions and token probabilities
    print("\nRunning classification evaluation...")
    preds, labels, probs, token_probs, token_valid, token_rats = get_predictions(
        model, test_loader, device
    )

    # Classification metrics
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    try:
        labels_bin = label_binarize(labels, classes=[0,1,2])
        auroc = roc_auc_score(labels_bin, probs, average='macro', multi_class='ovr')
    except:
        auroc = 0.5

    print("\nTest Set Performance:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Macro F1:  {macro_f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  AUROC:     {auroc:.4f}")
    print("\n", classification_report(labels, preds, target_names=['Normal', 'Offensive', 'Hate speech'], digits=4))

    # Error cases
    texts = [' '.join(tokens) for tokens in test_words]
    save_error_cases(preds, labels, probs, texts)

    # Explainability
    explainability_results = None
    if not args.no_explainability:
        explainability_results = compute_explainability_metrics(
            model, test_loader, device, tokenizer,
            preds, labels, probs,
            token_probs, token_valid, token_rats
        )

    # Bias metrics
    bias_results = None
    if not args.no_bias and any(t is not None for t in test_targets):
        bias_results = compute_bias_metrics(model, test_loader, device, test_targets)
    elif not args.no_bias:
        print("\nSkipping bias metrics: no target categories found.")

    # Summary
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"F1: {macro_f1:.4f}  |  Accuracy: {acc:.4f}  |  AUROC: {auroc:.4f}")
    if explainability_results:
        p = explainability_results['plausibility']
        f = explainability_results['faithfulness']
        print(f"Plausibility: Token F1={p['token_f1']:.3f}  Span IOU F1={p['span_iou_f1']:.3f}  AUPRC={p['auprc']:.3f}")
        print(f"Faithfulness: Comp={f['comprehensiveness']:.3f}  Suff={f['sufficiency']:.3f}")
    if bias_results:
        g = bias_results['gmb_metrics']
        print(f"Bias: Overall AUC={bias_results['overall_auc']:.4f}  GMB Sub={g['gmb_subgroup_auc']:.4f}  "
              f"BPSN={g['gmb_bpsn_auc']:.4f}  BNSP={g['gmb_bnsp_auc']:.4f}")

    if args.save_results:
        save_data = {
            'accuracy': acc,
            'macro_f1': macro_f1,
            'precision': precision,
            'recall': recall,
            'auroc': auroc,
            'predictions': preds.tolist(),
            'labels': labels.tolist(),
            'probabilities': probs.tolist(),
        }
        if explainability_results:
            save_data['explainability'] = explainability_results
        if bias_results:
            save_data['bias'] = bias_results
        with open(args.save_results, 'w') as f:
            json.dump(save_data, f, indent=2)
        print(f"Results saved to {args.save_results}")

if __name__ == "__main__":
    main()

Using device: cuda
Loading and preparing test split...
Test samples: 1923
Loading model from /content/gdrive/MyDrive/models/CAP_model(lambda=5)/best_cap_model.pt


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Running classification evaluation...


Predicting: 100%|██████████| 121/121 [00:15<00:00,  8.01it/s]



Test Set Performance:
  Accuracy:  0.6916
  Macro F1:  0.6811
  Precision: 0.6809
  Recall:    0.6814
  AUROC:     0.8498

               precision    recall  f1-score   support

      Normal     0.7531    0.7475    0.7503       800
   Offensive     0.5348    0.5368    0.5358       544
 Hate speech     0.7547    0.7599    0.7573       579

    accuracy                         0.6916      1923
   macro avg     0.6809    0.6814    0.6811      1923
weighted avg     0.6919    0.6916    0.6917      1923

Saved 593 error cases to error_cases.json

Computing Explainability Metrics (CAP model)


Faithfulness: 100%|██████████| 121/121 [00:40<00:00,  2.98it/s]



Examples evaluated: 1122

Plausibility:
  Token F1:          0.722
  Token IOU:         0.628
  Span IOU F1:       0.591
  AUPRC:             0.778

Faithfulness:
  Comprehensiveness: 0.422
  Sufficiency:       -0.060

Computing Bias Metrics (HateXplain Method)


Bias evaluation: 100%|██████████| 121/121 [00:14<00:00,  8.34it/s]



Overall AUC: 0.8703
African         310      0.8593       0.7401       0.9300      
Islam           197      0.8737       0.8317       0.8950      
Jewish          185      0.8950       0.7061       0.9524      
Homosexual      177      0.8812       0.8602       0.8881      
Women           155      0.7280       0.8437       0.8033      
Refugee         81       0.7064       0.9012       0.7062      
Arab            64       0.5000       0.5000       0.9373      
Caucasian       43       0.8437       0.9525       0.6910      
Asian           39       0.9222       0.9473       0.7909      
Hispanic        27       1.0000       0.9722       0.9439      

GMB (p=-5): Subgroup=0.7107  BPSN=0.7121  BNSP=0.8184

SUMMARY
F1: 0.6811  |  Accuracy: 0.6916  |  AUROC: 0.8498
Plausibility: Token F1=0.722  Span IOU F1=0.591  AUPRC=0.778
Faithfulness: Comp=0.422  Suff=-0.060
Bias: Overall AUC=0.8703  GMB Sub=0.7107  BPSN=0.7121  BNSP=0.8184
